# Baseline 3 — StyleGAN2 (+ differentiable augmentation) on Batik_Lasem (canonical 50% subset)

**Project:** Deep GANs for Aesthetic-Driven Apparel Pattern Synthesis.

### Why this uses `lucidrains/stylegan2_pytorch` instead of the NVLabs repo
The official **NVIDIA StyleGAN2-ADA** repo is a 2021 codebase whose custom CUDA
kernels (`bias_act`, `upfirdn2d`) **do not compile** on current Colab
(PyTorch 2.11 / CUDA 12.8), and `train.py` crashes on the modern PyTorch API. On an
L4 (Ada, sm_89) you also cannot downgrade PyTorch. The task explicitly allows
*"a well-established compatible implementation"*, so we use the widely-used,
pip-installable, **pure-PyTorch** `lucidrains/stylegan2_pytorch`, which:
* needs **no custom-CUDA compilation** (runs on PyTorch 2.x / L4), and
* includes **differentiable augmentation** (translation + cutout) for limited data
  — the practical equivalent of ADA's adaptive augmentation.

**Comparability is preserved (critical):**
* Trains on the **exact same** canonical 50% TRAIN split, pre-resized to 128×128
  with the identical bicubic preprocessing (`data/splits/batik_lasem_50pct_train.csv`).
* The held-out **TEST** split is **never** used for training.
* FID/KID are computed with the **same `batik_gan.metrics` evaluator + test set**
  as DCGAN and cWGAN-GP, so the three baselines are directly comparable.
* Same output layout (`outputs/StyleGAN2_ADA/`), `history.csv`, 300-dpi plots,
  checkpoint/resume.

> Augmentation is geometric only (translation, cutout) — **no colour augmentation**,
> to preserve motif colour semantics.


## 1. Configuration

In [ ]:

# =====================================================================
# CONFIGURATION
# =====================================================================
import math
CONFIG = {
    "model_name": "StyleGAN2_ADA",     # keep folder name for the comparison notebook
    "implementation": "lucidrains/stylegan2_pytorch (pure PyTorch, DiffAugment)",
    "seed": 42,
    "image_size": 128,
    "latent_dim": 512,
    "network_capacity": 16,
    "batch_size": 16,                  # raise to 24/32 if VRAM allows on L4
    "gradient_accumulate_every": 4,    # effective batch = batch_size * this
    "learning_rate": 2e-4,
    "aug_prob": 0.5,                   # differentiable-augmentation probability
    "aug_types": ["translation", "cutout"],   # geometric only (preserve colour)
    "total_steps": 30000,              # training length in optimiser steps
    "steps_per_epoch": 500,            # one "epoch" = this many steps (for eval/logging)
    "eval_n_gen": 640,                 # generated samples per held-out evaluation
    "eval_kid_subset": 100,
    "n_fixed": 64,                     # fixed-noise grid size
    "num_workers": 2,
    "pin_version": "1.8.10",           # stylegan2_pytorch version
    "resume": True,
}
CONFIG["num_layers"] = int(math.log2(CONFIG["image_size"])) - 1   # 128 -> 6


## 2. Colab setup (clone repo + install stylegan2_pytorch)

In [ ]:

# =====================================================================
# COLAB SETUP: clone our repo (src/ + splits) + install stylegan2_pytorch
#   We install with --no-deps so pip does NOT downgrade the Colab PyTorch,
#   then add the library's lightweight runtime deps explicitly.
# =====================================================================
import os, sys, subprocess, glob, json, time

REPO_URL = "https://github.com/sid-2k6/Textile_Pattern_GAN.git"
REPO_DIR = "/content/Textile_Pattern_GAN"
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "gan/baseline-notebooks",
                    REPO_URL, REPO_DIR], check=False)
    if not os.path.isdir(os.path.join(REPO_DIR, ".git")):   # branch clone fallback
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                f"stylegan2_pytorch=={CONFIG['pin_version']}"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "kornia", "einops", "retry", "tqdm", "pillow",
                "torchmetrics>=1.0.0", "torch-fidelity", "scipy"], check=False)

# sanity import
try:
    import stylegan2_pytorch
    from stylegan2_pytorch import Trainer
    print("stylegan2_pytorch OK:", getattr(stylegan2_pytorch, "__version__", "?"))
except Exception as e:
    import traceback; traceback.print_exc()
    print("If import failed, re-run this cell (kornia/einops may need a fresh import).")
print("Setup complete.")


## 3. Environment verification

In [ ]:

# =====================================================================
# ENVIRONMENT VERIFICATION
# =====================================================================
from batik_gan import env
ENV_INFO = env.print_environment()


## 4. Imports + reproducibility

In [ ]:

# =====================================================================
# IMPORTS + REPRODUCIBILITY
# =====================================================================
import numpy as np, pandas as pd, torch
from batik_gan import env, paths as P, data as D, metrics as M, viz, manifest as MAN
from batik_gan.checkpoint import HistoryLogger
from stylegan2_pytorch import Trainer
from stylegan2_pytorch.stylegan2_pytorch import styles_def_to_tensor, image_noise
try:
    from stylegan2_pytorch.stylegan2_pytorch import NanException
except Exception:
    class NanException(Exception):
        pass
env.set_seed(CONFIG["seed"], deterministic=True)
DEVICE = env.get_device()
print("Device:", DEVICE)


## 5. Google Drive

In [ ]:

# =====================================================================
# GOOGLE DRIVE MOUNT
# =====================================================================
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception as e:
    IN_COLAB = False
    print("Not running in Colab (or already mounted):", e)


## 6. Paths + validation

In [ ]:

# =====================================================================
# PATH CONFIGURATION  --- EDIT DATASET_ROOT TO MATCH YOUR DRIVE ---
# =====================================================================
DRIVE_ROOT    = "/content/drive/MyDrive"
DATASET_ROOT  = f"{DRIVE_ROOT}/hari/Textile_Pattern_GAN/Datasets"          # <-- EDIT if needed
METADATA_PATH = f"{DATASET_ROOT}/Batik_Lasem/motifs (isen-isen)/metadata motifs.csv"
OUTPUT_ROOT   = f"{DRIVE_ROOT}/hari/Textile_Pattern_GAN/outputs"
SPLITS_DIR    = os.path.join(REPO_DIR, "data", "splits")

paths = P.ProjectPaths(project_root=REPO_DIR, dataset_root=DATASET_ROOT,
                       metadata_path=METADATA_PATH, splits_dir=SPLITS_DIR,
                       output_root=OUTPUT_ROOT, model_name=CONFIG["model_name"]).make_dirs()
need_meta = not os.path.isfile(os.path.join(SPLITS_DIR, "batik_lasem_50pct_train.csv"))
P.validate_paths(paths, require_metadata=need_meta)

# lucidrains Trainer writes checkpoints under SG2_BASE/models/<NAME>/
SG2_BASE = os.path.join(OUTPUT_ROOT, "StyleGAN2_ADA")
SG2_NAME = "batik_sg2"
STAGE_DIR = "/content/sg2_stage_train"       # local folder of 128px training PNGs


## 7. Load shared 50% split + audit

In [ ]:

# =====================================================================
# LOAD THE SHARED 50% SPLIT (same as DCGAN / cWGAN-GP) + AUDIT
# =====================================================================
train_csv = os.path.join(SPLITS_DIR, "batik_lasem_50pct_train.csv")
test_csv  = os.path.join(SPLITS_DIR, "batik_lasem_50pct_test.csv")
if not (os.path.isfile(train_csv) and os.path.isfile(test_csv)):
    MAN.build_all(METADATA_PATH, SPLITS_DIR, frac=0.5, test_frac=0.2, seed=CONFIG["seed"])
train_df = pd.read_csv(train_csv); test_df = pd.read_csv(test_csv)
AUDIT = D.audit_dataset(train_df, test_df, DATASET_ROOT, sample_check=400, stop_on_error=True)


## 8. Stage 128px training images

In [ ]:

# =====================================================================
# STAGE 128x128 TRAIN PNGs (identical preprocessing; TEST is held out)
# lucidrains reads an image folder directly — no dataset_tool needed.
# =====================================================================
from PIL import Image
from batik_gan.paths import resolve_image_path
os.makedirs(STAGE_DIR, exist_ok=True)
existing = len(glob.glob(os.path.join(STAGE_DIR, "*.png")))
if existing < len(train_df):
    print("Staging 128x128 training PNGs ...")
    n_ok = 0
    for i, r in train_df.reset_index(drop=True).iterrows():
        ap = resolve_image_path(DATASET_ROOT, r.get("rel_path", ""), r["filename"])
        if ap is None:
            continue
        Image.open(ap).convert("RGB").resize(
            (CONFIG["image_size"], CONFIG["image_size"]), Image.BICUBIC
        ).save(os.path.join(STAGE_DIR, f"{i:05d}.png"))
        n_ok += 1
    print(f"Staged {n_ok} images -> {STAGE_DIR}")
else:
    print(f"Already staged {existing} images in {STAGE_DIR}")


## 9. Held-out evaluator + fixed noise

In [ ]:

# =====================================================================
# HELD-OUT EVALUATOR (FID/KID/diversity) + FIXED EVAL NOISE
# REAL = test set; identical protocol to DCGAN / cWGAN-GP.
# =====================================================================
test_ds = D.BatikCanonicalDataset(test_df, DATASET_ROOT, CONFIG["image_size"],
                                  conditional=False, verify=True)
test_loader = D.make_dataloader(test_ds, CONFIG["batch_size"], shuffle=False,
                                num_workers=CONFIG["num_workers"], seed=CONFIG["seed"],
                                drop_last=False)
real_uint8 = M.build_real_uint8_from_loader(test_loader)
print("Real (test) images for FID/KID:", tuple(real_uint8.shape))
evaluator = M.GenerativeEvaluator(real_uint8, DEVICE, kid_subset_size=CONFIG["eval_kid_subset"])

# fixed latent + fixed per-pixel noise -> comparable epoch grids
g = torch.Generator(device="cpu").manual_seed(CONFIG["seed"])
fixed_z = torch.randn(CONFIG["n_fixed"], CONFIG["latent_dim"], generator=g).to(DEVICE)
fixed_inoise = image_noise(CONFIG["n_fixed"], CONFIG["image_size"], device=DEVICE)
torch.save({"z": fixed_z.cpu()}, os.path.join(paths.samples_dir, "fixed_noise.pt"))


## 10. Build Trainer + resume

In [ ]:

# =====================================================================
# BUILD TRAINER + RESUME
# =====================================================================
trainer = Trainer(
    name=SG2_NAME, base_dir=SG2_BASE,
    image_size=CONFIG["image_size"],
    network_capacity=CONFIG["network_capacity"],
    batch_size=CONFIG["batch_size"],
    gradient_accumulate_every=CONFIG["gradient_accumulate_every"],
    lr=CONFIG["learning_rate"],
    num_workers=CONFIG["num_workers"],
    aug_prob=CONFIG["aug_prob"],
    aug_types=CONFIG["aug_types"],
    save_every=10**9, evaluate_every=10**9,   # disable library's own save/eval; we drive it
    calculate_fid_every=None,
)
trainer.set_data_src(STAGE_DIR)

HISTORY_COLUMNS = ["epoch", "step", "generator_loss", "discriminator_loss",
                   "fid", "kid_mean", "kid_std", "diversity",
                   "lr_g", "lr_d", "epoch_time", "gpu_memory_mb"]
history = HistoryLogger(paths.history_csv, HISTORY_COLUMNS)

models_path = os.path.join(SG2_BASE, "models", SG2_NAME)
have_ckpt = os.path.isdir(models_path) and len(glob.glob(os.path.join(models_path, "model_*.pt"))) > 0
start_epoch = history.last_epoch()
best_fid, best_epoch = float("inf"), -1
if history.rows:
    fids = [r.get("fid") for r in history.rows if r.get("fid") == r.get("fid")]
    if fids:
        best_fid = min(fids); best_epoch = int(history.rows[int(np.argmin(
            [r.get("fid", float("inf")) for r in history.rows]))]["epoch"])

if CONFIG["resume"] and have_ckpt and start_epoch > 0:
    try:
        trainer.load(-1)
        trainer.steps = start_epoch * CONFIG["steps_per_epoch"]
        print(f"Checkpoint found. Resuming from epoch {start_epoch} "
              f"(best FID so far {best_fid:.2f} @ epoch {best_epoch}).")
    except Exception as e:
        print("Resume failed, starting fresh:", e); start_epoch = 0
else:
    print("No checkpoint found. Starting from epoch 0.")


## 11. Generation helpers

In [ ]:

# =====================================================================
# GENERATION HELPERS (EMA generator; matches library inference path)
# =====================================================================
@torch.no_grad()
def gen_from(z, inoise):
    trainer.GAN.eval()
    w = trainer.GAN.SE(z)                               # EMA style vectorizer
    w_tensor = styles_def_to_tensor([(w, CONFIG["num_layers"])])
    imgs = trainer.GAN.GE(w_tensor, inoise)            # EMA generator
    return imgs.clamp_(0., 1.)                         # [0,1]

def sample_generator(n):
    z = torch.randn(n, CONFIG["latent_dim"], device=DEVICE)
    ni = image_noise(n, CONFIG["image_size"], device=DEVICE)
    return gen_from(z, ni)


## 12. Training loop

In [ ]:

# =====================================================================
# TRAINING LOOP (epoch = steps_per_epoch optimiser steps)
#   * NaN steps are skipped (not fabricated)
#   * held-out FID/KID + fixed-noise grid + checkpoint each epoch
#   * old library checkpoints pruned (keep last 3 + best) to bound Drive usage
# =====================================================================
from batik_gan.train import epoch_summary_print

def prune_models(keep):
    for p in glob.glob(os.path.join(models_path, "model_*.pt")):
        try:
            num = int(os.path.basename(p).split("_")[1].split(".")[0])
            if num not in keep:
                os.remove(p)
        except Exception:
            pass

EPOCHS = max(1, CONFIG["total_steps"] // CONFIG["steps_per_epoch"])
for epoch in range(start_epoch + 1, EPOCHS + 1):
    env.reset_peak_memory()
    t0 = time.time(); gl, dl = [], []
    for _ in range(CONFIG["steps_per_epoch"]):
        try:
            trainer.train()
        except NanException:
            print("NaN encountered — skipping step"); continue
        gl.append(float(trainer.g_loss)); dl.append(float(trainer.d_loss))

    eval_metrics = evaluator.evaluate(sample_generator, n_gen=CONFIG["eval_n_gen"],
                                      batch=CONFIG["batch_size"], compute_diversity=True)
    grid = gen_from(fixed_z, fixed_inoise)
    viz.save_sample_grid(grid, os.path.join(paths.samples_dir, f"epoch_{epoch:04d}.png"),
                         title=f"StyleGAN2 epoch {epoch}")

    trainer.save(epoch)                       # library checkpoint (model_{epoch}.pt)
    fid = eval_metrics.get("fid", float("nan"))
    if fid == fid and fid < best_fid:
        best_fid, best_epoch = fid, epoch
    keep = {best_epoch} | set(range(max(1, epoch - 2), epoch + 1))
    prune_models(keep)

    epoch_time = time.time() - t0; gpu_mb = env.gpu_memory_mb()
    row = {"epoch": epoch, "step": int(trainer.steps),
           "generator_loss": float(np.mean(gl)) if gl else float("nan"),
           "discriminator_loss": float(np.mean(dl)) if dl else float("nan"),
           "fid": fid, "kid_mean": eval_metrics.get("kid_mean", float("nan")),
           "kid_std": eval_metrics.get("kid_std", float("nan")),
           "diversity": eval_metrics.get("diversity", float("nan")),
           "lr_g": CONFIG["learning_rate"], "lr_d": CONFIG["learning_rate"],
           "epoch_time": epoch_time, "gpu_memory_mb": gpu_mb}
    history.append(row)

    epoch_summary_print(
        "StyleGAN2", epoch, EPOCHS,
        {"Generator Loss": round(row["generator_loss"], 4),
         "Discriminator Loss": round(row["discriminator_loss"], 4),
         "Steps": int(trainer.steps)},
        eval_metrics, {"Generator": row["lr_g"], "Discriminator": row["lr_d"]},
        epoch_time, {"model": os.path.join(models_path, f"model_{epoch}.pt"),
                     "best_epoch": best_epoch}, gpu_mb)

print("Training complete. Best FID", best_fid, "@ epoch", best_epoch)


## 13. Plots

In [ ]:

# =====================================================================
# PUBLICATION PLOTS (font 20, dpi 300)
# =====================================================================
hist_df = pd.read_csv(paths.history_csv)
made = viz.plot_history(hist_df, paths.plots_dir, "StyleGAN2", has_gp=False)
print("Saved plots:")
for m in made: print("  ", m)


## 14. Final evaluation (held-out test set)

In [ ]:

# =====================================================================
# FINAL EVALUATION on the held-out TEST set (best checkpoint)
# =====================================================================
try:
    trainer.load(best_epoch); loaded = f"model_{best_epoch}.pt"
except Exception:
    trainer.load(-1); loaded = "latest"
final = evaluator.evaluate(sample_generator, n_gen=CONFIG["eval_n_gen"],
                           batch=CONFIG["batch_size"], compute_diversity=True)
with torch.no_grad():
    viz.save_sample_grid(gen_from(fixed_z, fixed_inoise),
                         os.path.join(paths.samples_dir, "final_grid.png"),
                         title=f"StyleGAN2 final (best epoch {best_epoch})")

hist_df = pd.read_csv(paths.history_csv)
n_params = sum(p.numel() for p in trainer.GAN.GE.parameters())
final_metrics = {
    "model": "StyleGAN2_ADA",
    "implementation": CONFIG["implementation"],
    "best_epoch": int(best_epoch), "best_checkpoint": loaded,
    "best_fid": float(best_fid) if best_fid != float("inf") else float("nan"),
    "final_fid": final["fid"], "final_kid": final["kid_mean"],
    "final_kid_std": final["kid_std"], "final_diversity": final["diversity"],
    "final_generator_loss": float(pd.to_numeric(hist_df["generator_loss"], errors="coerce").dropna().iloc[-1])
        if hist_df["generator_loss"].notna().any() else float("nan"),
    "final_discriminator_loss": float(pd.to_numeric(hist_df["discriminator_loss"], errors="coerce").dropna().iloc[-1])
        if hist_df["discriminator_loss"].notna().any() else float("nan"),
    "training_time_sec": float(pd.to_numeric(hist_df["epoch_time"], errors="coerce").sum()),
    "peak_gpu_memory_mb": float(pd.to_numeric(hist_df["gpu_memory_mb"], errors="coerce").max()),
    "generator_parameters": int(n_params),
    "total_steps_trained": int(trainer.steps),
    "n_real_test": int(real_uint8.shape[0]), "n_gen_eval": CONFIG["eval_n_gen"],
    "augmentation": f"DiffAugment {CONFIG['aug_types']} p={CONFIG['aug_prob']}",
    "real_accuracy": "N/A (StyleGAN2 logistic loss; no fixed accuracy)",
    "generator_accuracy": "N/A (not defined for GANs)",
}
with open(paths.final_metrics_json, "w") as f:
    json.dump(final_metrics, f, indent=2, default=str)
pd.DataFrame([final_metrics]).to_csv(paths.final_metrics_csv, index=False)
print(json.dumps(final_metrics, indent=2, default=str))


## 15. Final results summary

In [ ]:

# =====================================================================
# FINAL RESULTS SUMMARY
# =====================================================================
print("="*60); print("StyleGAN2 (lucidrains, DiffAugment) — FINAL SUMMARY"); print("="*60)
for k in ("best_epoch", "best_fid", "final_fid", "final_kid", "final_diversity",
          "generator_parameters", "total_steps_trained",
          "training_time_sec", "peak_gpu_memory_mb"):
    print(f"{k:26s}: {final_metrics.get(k)}")
print("FID/KID use the SAME evaluator + held-out test set as DCGAN/cWGAN-GP")
print("-> directly comparable in notebook 04.")
print("="*60)
